# Session 2.1 -- Embeddings, Choosing the Correct Model

**AI-2: AI Backend Engineering**

---

## Learning Objectives

By the end of this session, you will be able to:

1. **Explain** what embeddings are and how they represent semantic meaning as numerical vectors
2. **Compare** at least two embedding models on a controlled retrieval task using measurable metrics
3. **Justify** a model selection decision with data, articulating tradeoffs across the dimensions-cost-quality triangle

---

## What You'll Need

- A working `ai2-pipeline/` project (cloned and `uv sync` completed)
- Your Voyage AI API key in a `.env` file at the project root
- This notebook open in Jupyter (via VS Code, JupyterLab, etc.)

## How This Notebook Works

This is a **walk-along notebook**. It mirrors the slides and demos being presented. Run each cell as the instructor walks through the material. Experiment in the "Try it yourself" cells -- they are yours to modify and re-run.

---

## 1. Setup

Before we can generate embeddings, we need two things:
1. Our project dependencies installed (`uv sync` -- already done)
2. Our Voyage AI API key loaded from `.env`

The cell below sets up the Python path so we can import from `src/` and loads environment variables.

In [ ]:
import sys
sys.path.insert(0, "../..")

from dotenv import load_dotenv
load_dotenv()

Let's verify your Voyage AI key is loaded. You should see a confirmation message -- **not** the key itself.

In [ ]:
import os

key = os.getenv("VOYAGE_API_KEY")
if key and key.startswith("pa-"):
    print(f"Voyage AI key loaded successfully (starts with {key[:6]}...)")
elif key:
    print(f"Key found but format looks unexpected. Check your Voyage AI key.")
else:
    print("ERROR: No VOYAGE_API_KEY found. Check your .env file.")

Now import the pre-built embedding module. This is `src/s2_embeddings/embed.py` -- it is **provided complete** as course infrastructure. You will use it throughout this notebook and future sessions.

In [ ]:
from src.s2_embeddings.embed import get_embedding, embed_texts, cosine_similarity

print("Imports successful -- ready to generate embeddings.")

We will also need these libraries for visualization and analysis later in the notebook.

In [ ]:
import numpy as np
import pathlib
import time
import json
import os
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Embedding cache -- avoids re-calling the API on re-runs
# To force re-embedding, delete the cache directory: rm -rf cache/embeddings
CACHE_DIR = pathlib.Path("cache/embeddings")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("All imports loaded.")
print(f"Embedding cache directory: {CACHE_DIR.resolve()}")

---

## 2. What Are Embeddings?

Think of **GPS coordinates**. New York and New Jersey are close on a map -- their coordinates are similar. New York and Tokyo are far apart.

Embeddings do the same thing for **meaning**:
- "Budget report" and "financial summary" are **close**
- "Budget report" and "lunch menu" are **far apart**

An embedding model takes text and returns a **fixed-length list of numbers**. That list is the text's **address in meaning-space**.

```
"The quarterly budget report shows a 12% increase"
                    |
              [Embedding Model]
                    |
        [0.023, -0.041, 0.089, 0.012, ...]
                 (1024 numbers)
```

Let's see this in action.

### Your first embedding

One API call. You get back a list of numbers.

In [ ]:
# Embed a single sentence
vector = get_embedding("The quarterly budget report shows a 12% increase")

print(f"Dimensions: {len(vector)}")
print(f"First 10 values: {vector[:10]}")
print(f"Type: {type(vector)}")

That is the entire embedding. A list of 1024 floating point numbers. This list **is** the meaning of that sentence, encoded as a point in 1024-dimensional space.

Important: the embedding model and Claude are **different models**. Claude generates text. The embedding model generates vectors. They serve different purposes in our pipeline.

---

## 3. Semantic Similarity

If similar meanings produce similar vectors, we can **measure** similarity. The tool is **cosine similarity** -- it computes the angle between two vectors and returns a score from -1 to 1. Higher means more similar.

Let's embed three sentences and compare them.

In [ ]:
# Embed three sentences
texts = [
    "The quarterly budget report shows a 12% increase",
    "Financial summary indicates revenue growth of 12 percent",
    "The office lunch menu for Tuesday features pasta"
]

vectors = embed_texts(texts)

print(f"Budget vs Financial: {cosine_similarity(vectors[0], vectors[1]):.4f}")
print(f"Budget vs Lunch:     {cosine_similarity(vectors[0], vectors[2]):.4f}")
print(f"Financial vs Lunch:  {cosine_similarity(vectors[1], vectors[2]):.4f}")

The two financial sentences score high -- they mean similar things despite using different words. The lunch menu scores low against both. **The model captured meaning, not just keywords.**

Think about this: "Budget report" and "financial summary" share **zero words**, but the embedding model knows they are about the same thing.

### Try it yourself

Type in your own sentences below. Try pairs that are:
- Semantically similar but use different words
- About completely different topics
- Surprisingly similar or surprisingly different

What scores do you get?

In [ ]:
# Your turn -- change these sentences to anything you want
sentence_a = "Machine learning is transforming how businesses operate"
sentence_b = "AI is changing the way companies do things"
sentence_c = "I went to the grocery store to buy apples"

vec_a = get_embedding(sentence_a)
vec_b = get_embedding(sentence_b)
vec_c = get_embedding(sentence_c)

print(f"A vs B: {cosine_similarity(vec_a, vec_b):.4f}  (should be high)")
print(f"A vs C: {cosine_similarity(vec_a, vec_c):.4f}  (should be low)")
print(f"B vs C: {cosine_similarity(vec_b, vec_c):.4f}  (should be low)")

### Checkpoint 1 -- Verify your setup

Run this cell. If AI vs ML scores higher than AI vs cake, your embedding module is working.

In [ ]:
v1 = get_embedding("artificial intelligence")
v2 = get_embedding("machine learning")
v3 = get_embedding("chocolate cake recipe")

sim_ai_ml = cosine_similarity(v1, v2)
sim_ai_cake = cosine_similarity(v1, v3)

print(f"AI vs ML:   {sim_ai_ml:.4f}")
print(f"AI vs Cake: {sim_ai_cake:.4f}")
print()

if sim_ai_ml > sim_ai_cake:
    print("Checkpoint 1 PASSED -- your embedding setup is working.")
else:
    print("Something is off. Flag the instructor.")

---

## 4. Semantic Cluster Scatter Plot

When you embed many phrases, **patterns emerge**. Phrases about finance cluster together. Phrases about food cluster together. Each topic occupies its own neighborhood in embedding space.

We cannot visualize 1024 dimensions directly, but we can use **PCA** (Principal Component Analysis) to reduce the vectors down to 2 dimensions for plotting. This loses detail, but the clusters remain visible.

Let's embed a curated set of phrases from different categories and see if the embedding model groups them the way we would.

In [ ]:
# Curated phrases across 4 categories
cluster_data = {
    "Finance": [
        "Quarterly revenue exceeded projections by 8 percent",
        "The annual budget allocation was approved by the board",
        "Net income grew year over year driven by new clients",
        "Cash reserves remain healthy at 8.2 million",
        "Operating expenses as a percentage of revenue declined"
    ],
    "HR / Policy": [
        "All employees receive 20 vacation days per year",
        "The hybrid work schedule requires 3 days in office",
        "Performance reviews are conducted quarterly by managers",
        "New hires are paired with a peer mentor for 120 days",
        "Professional development budget is 3200 dollars annually"
    ],
    "Technology / IT": [
        "All employees must enable multi-factor authentication",
        "The VPN client is required for remote access to systems",
        "Device compliance checks begin March 1",
        "Password rotation is required every 90 days",
        "The cloud migration project completes mid 2025"
    ],
    "Food / Misc": [
        "The lunch menu features pasta and salad today",
        "We are ordering pizza for the team celebration",
        "The coffee machine on the third floor is broken",
        "Friday afternoon snacks are in the break room",
        "The catering company confirmed delivery for noon"
    ]
}

# Flatten for embedding
all_phrases = []
all_labels = []
for category, phrases in cluster_data.items():
    for phrase in phrases:
        all_phrases.append(phrase)
        all_labels.append(category)

# Cache cluster embeddings to avoid re-calling the API on notebook re-runs
cache_file = CACHE_DIR / f"cluster_phrases_{len(all_phrases)}_chunks.npz"

if cache_file.exists():
    data = np.load(cache_file)
    cluster_vectors = data["embeddings"].tolist()
    elapsed = float(data["elapsed"])
    print(f"Cluster embeddings: loaded from cache ({len(all_phrases)} phrases, dim={len(cluster_vectors[0])}, original time: {elapsed:.2f}s)")
else:
    print(f"Embedding {len(all_phrases)} phrases across {len(cluster_data)} categories...")
    start = time.time()
    cluster_vectors = embed_texts(all_phrases)
    elapsed = time.time() - start
    np.savez(cache_file, embeddings=np.array(cluster_vectors), elapsed=np.array(elapsed))
    print(f"Done in {elapsed:.2f}s — cached to {cache_file.name}")

In [ ]:
# Reduce 1024 dimensions to 2 using PCA
pca = PCA(n_components=2)
coords_2d = pca.fit_transform(np.array(cluster_vectors))

print(f"PCA explained variance: {pca.explained_variance_ratio_[0]:.1%} + {pca.explained_variance_ratio_[1]:.1%} = {sum(pca.explained_variance_ratio_):.1%}")
print("(This is how much structure the 2D projection captures from the original 1024 dimensions.)")

In [ ]:
# Plot the clusters
category_colors = {
    "Finance": "#2196F3",
    "HR / Policy": "#4CAF50",
    "Technology / IT": "#FF9800",
    "Food / Misc": "#E91E63"
}

fig, ax = plt.subplots(figsize=(10, 7))

for category in cluster_data.keys():
    mask = [label == category for label in all_labels]
    xs = coords_2d[mask, 0]
    ys = coords_2d[mask, 1]
    ax.scatter(xs, ys, label=category, color=category_colors[category], s=80, alpha=0.8, edgecolors="white", linewidth=0.5)

# Annotate each point with a short version of the phrase
for i, phrase in enumerate(all_phrases):
    short = phrase[:35] + "..." if len(phrase) > 35 else phrase
    ax.annotate(short, (coords_2d[i, 0], coords_2d[i, 1]),
                fontsize=6, alpha=0.7,
                xytext=(5, 5), textcoords="offset points")

ax.set_title("Semantic Clusters -- Embedding Space (PCA to 2D)", fontsize=13)
ax.set_xlabel("PCA Component 1")
ax.set_ylabel("PCA Component 2")
ax.legend(loc="best", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

You should see phrases from the same category clustering together. Finance phrases land near finance phrases. Food phrases land near food phrases. The embedding model organized text by **meaning** without being told what the categories were.

Note: PCA is lossy -- we compressed 1024 dimensions into 2. The actual embedding space preserves more structure than this plot shows.

### Try it yourself -- Add your own phrase

Type a phrase below and see where it lands in the scatter plot. Try phrases that:
- Clearly belong to one category
- Sit between categories (e.g., "food delivery app" -- food or tech?)
- Are from a category not on the chart

In [ ]:
# Type your own phrase here
my_phrase = "The expense reimbursement policy covers meals up to 65 dollars per day"

# Embed it and project into the same 2D space
my_vector = get_embedding(my_phrase)
my_2d = pca.transform(np.array([my_vector]))

# Re-draw the plot with your phrase added
fig, ax = plt.subplots(figsize=(10, 7))

for category in cluster_data.keys():
    mask = [label == category for label in all_labels]
    xs = coords_2d[mask, 0]
    ys = coords_2d[mask, 1]
    ax.scatter(xs, ys, label=category, color=category_colors[category], s=80, alpha=0.8, edgecolors="white", linewidth=0.5)

# Plot your phrase as a star
ax.scatter(my_2d[0, 0], my_2d[0, 1], marker="*", s=300, color="red", zorder=5, label="YOUR PHRASE")
ax.annotate(my_phrase[:50], (my_2d[0, 0], my_2d[0, 1]),
            fontsize=8, fontweight="bold", color="red",
            xytext=(10, 10), textcoords="offset points")

ax.set_title("Where does YOUR phrase land?", fontsize=13)
ax.set_xlabel("PCA Component 1")
ax.set_ylabel("PCA Component 2")
ax.legend(loc="best", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Also show similarity to each category centroid
print("\nSimilarity to each category (average):")
for category, phrases in cluster_data.items():
    cat_indices = [i for i, l in enumerate(all_labels) if l == category]
    cat_vecs = [cluster_vectors[i] for i in cat_indices]
    avg_sim = np.mean([cosine_similarity(my_vector, cv) for cv in cat_vecs])
    print(f"  {category:<20} {avg_sim:.4f}")

---

## -- Break (15 minutes) --

When we return: **How to measure embedding quality**. We shift from "I can generate embeddings" to "I can evaluate them."

---

## 5. Cross-Model Comparison

You can generate embeddings now. But how do you know if they are **good**? Good is not a feeling. Good is a **number**.

We evaluate embeddings on the task they serve: **retrieval**. Can this embedding model help us find the right document for a given question?

The comparison framework is simple:
- **Same** documents
- **Same** queries
- **Same** retrieval method
- **Different** model

That is the only variable.

### Voyage AI Models

| Model | Dimensions | Tier | Cost/1k tokens |
|-------|-----------|------|---------------|
| `voyage-3-lite` | 1024 | Lite | $0.02 |
| `voyage-3` | 1024 | Standard | $0.06 |
| `voyage-3-large` | 2048 | Large | $0.18 |

### Model configuration

Pick at least 2 of the 3 models below. If you want the biggest contrast, compare `voyage-3-lite` with `voyage-3-large`.

In [ ]:
# Model metadata -- used for comparison table and cost estimates
MODELS = {
    "voyage-3-lite":  {"dimensions": 1024, "tier": "Lite",     "cost_per_1k_tokens": 0.02},
    "voyage-3":       {"dimensions": 1024, "tier": "Standard", "cost_per_1k_tokens": 0.06},
    "voyage-3-large": {"dimensions": 2048, "tier": "Large",    "cost_per_1k_tokens": 0.18},
}

# Choose which models to compare -- pick at least 2
SELECTED_MODELS = ["voyage-3-lite", "voyage-3", "voyage-3-large"]

print(f"Comparing {len(SELECTED_MODELS)} models: {', '.join(SELECTED_MODELS)}")

### Load the corpus

We will load a subset of the Northbrook Partners documents. Each document has an ID, a title, and text content.

In [ ]:
# Load Northbrook documents from the data directory
data_dir = Path("../../data/northbrook")

corpus = []
for filepath in sorted(data_dir.glob("*.md")):
    text = filepath.read_text()
    # Extract a title from the first heading
    first_line = text.strip().split("\n")[0].replace("# ", "").strip()
    doc_id = filepath.stem  # filename without extension
    # Use the first ~500 chars as our embedding text (keeps costs low for comparison)
    corpus.append({
        "id": doc_id,
        "title": first_line,
        "text": text[:500]
    })

print(f"Loaded {len(corpus)} documents:\n")
for doc in corpus:
    print(f"  {doc['id']:<40} {doc['title'][:55]}")

### Define test queries

Each query has a question and a list of document IDs that should be retrieved. These are our "known-good answers" -- the ground truth for evaluation.

Five queries are provided. You will add 2-3 of your own.

In [ ]:
test_queries = [
    {
        "query": "What is Northbrook's policy on remote work?",
        "relevant_doc_ids": ["remote_work_policy"]
    },
    {
        "query": "How does the quarterly performance review process work?",
        "relevant_doc_ids": ["employee_handbook"]
    },
    {
        "query": "What benefits and vacation days does Northbrook offer?",
        "relevant_doc_ids": ["vacation_policy_2025", "vacation_policy_2023", "employee_handbook"]
    },
    {
        "query": "What are the expense reimbursement limits for travel?",
        "relevant_doc_ids": ["expense_policy"]
    },
    {
        "query": "What security changes are required for all employees?",
        "relevant_doc_ids": ["memo_security_update"]
    },
    # -------------------------------------------------------
    # TODO: Add 2-3 of your own test queries below.
    # Look at the corpus document IDs above and write questions
    # that should retrieve specific documents.
    # -------------------------------------------------------
]

print(f"Test set: {len(test_queries)} queries")
for q in test_queries:
    print(f"  Q: {q['query'][:60]}")
    print(f"     Expected: {q['relevant_doc_ids']}")

### Embed the corpus with each model

We embed the same set of documents with each selected model and track how long it takes.

In [ ]:
def embed_corpus(documents, model):
    """Embed all documents with the given model. Returns dict: doc_id -> vector."""
    texts = [doc["text"] for doc in documents]
    vectors = embed_texts(texts, model=model)
    return {doc["id"]: vec for doc, vec in zip(documents, vectors)}

# Embed corpus with each model and track timing (with caching)
model_embeddings = {}
model_times = {}

for model_name in SELECTED_MODELS:
    safe_name = model_name.replace("-", "_")
    cache_file = CACHE_DIR / f"{safe_name}_{len(corpus)}_chunks.npz"

    if cache_file.exists():
        data = np.load(cache_file)
        stored_embeddings = data["embeddings"]
        elapsed = float(data["elapsed"])
        # Reconstruct the doc_id -> vector mapping
        model_embeddings[model_name] = {
            doc["id"]: stored_embeddings[i].tolist()
            for i, doc in enumerate(corpus)
        }
        model_times[model_name] = elapsed
        dims = stored_embeddings.shape[1]
        print(f"{model_name}: loaded from cache ({dims} dimensions, original time: {elapsed:.2f}s)")
    else:
        print(f"Embedding with {model_name}...", end=" ")
        start = time.time()
        model_embeddings[model_name] = embed_corpus(corpus, model_name)
        elapsed = time.time() - start
        model_times[model_name] = elapsed
        dims = len(list(model_embeddings[model_name].values())[0])
        # Save to cache
        ordered_vecs = [model_embeddings[model_name][doc["id"]] for doc in corpus]
        np.savez(cache_file, embeddings=np.array(ordered_vecs), elapsed=np.array(elapsed))
        print(f"done in {elapsed:.2f}s ({dims} dimensions) — cached")

print(f"\nAll {len(SELECTED_MODELS)} models complete.")
print(f"To force re-embedding, delete the cache: rm -rf {CACHE_DIR}")

---

## 6. Retrieval Evaluation

Now we test: given a query, does each model retrieve the correct documents?

**Top-k accuracy**: if we retrieve the top-k most similar documents, is the correct one among them? We use k=3.

Important: **do not compare raw similarity scores across models.** Different models produce different score distributions. Compare *which documents were retrieved*, not the raw numbers.

In [ ]:
def retrieve_top_k(query, corpus_embeddings, model, k=3):
    """Embed the query, find the k most similar documents."""
    query_vec = get_embedding(query, model=model)

    # Score every document
    scores = []
    for doc_id, doc_vec in corpus_embeddings.items():
        sim = cosine_similarity(query_vec, doc_vec)
        scores.append({"doc_id": doc_id, "score": sim})

    # Sort descending by score
    scores.sort(key=lambda x: x["score"], reverse=True)

    return scores[:k]


def evaluate_retrieval(queries, corpus_embeddings, model, k=3):
    """Run all test queries and compute top-k accuracy."""
    hits = 0
    results = []
    all_top_scores = []

    for q in queries:
        top_k = retrieve_top_k(q["query"], corpus_embeddings, model, k)
        retrieved_ids = [r["doc_id"] for r in top_k]
        all_top_scores.append(top_k[0]["score"])

        # Check if ANY relevant doc appears in top-k
        hit = any(rid in retrieved_ids for rid in q["relevant_doc_ids"])
        if hit:
            hits += 1

        results.append({
            "query": q["query"],
            "expected": q["relevant_doc_ids"],
            "retrieved": retrieved_ids,
            "hit": hit,
            "top_score": top_k[0]["score"]
        })

    return {
        "accuracy": hits / len(queries),
        "avg_top_score": np.mean(all_top_scores),
        "results": results
    }

print("Retrieval functions defined.")

In [ ]:
# Run evaluation for each model
model_evals = {}

for model_name in SELECTED_MODELS:
    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")

    eval_result = evaluate_retrieval(test_queries, model_embeddings[model_name], model_name)
    model_evals[model_name] = eval_result

    print(f"Top-3 Accuracy: {eval_result['accuracy']:.0%}")
    print(f"Avg Top Score:  {eval_result['avg_top_score']:.4f}")
    print()

    for r in eval_result["results"]:
        status = "HIT" if r["hit"] else "MISS"
        print(f"  [{status}] {r['query'][:50]}")
        print(f"         Expected:  {r['expected']}")
        print(f"         Retrieved: {r['retrieved']}")
        print(f"         Top score: {r['top_score']:.4f}")

### Comparison table

All the results side by side. This is the data you need to make a recommendation.

In [ ]:
# Build the comparison table
header = f"{'Metric':<25}"
for m in SELECTED_MODELS:
    header += f"{m:<20}"
print(header)
print("-" * (25 + 20 * len(SELECTED_MODELS)))

# Dimensions
row = f"{'Dimensions':<25}"
for m in SELECTED_MODELS:
    row += f"{MODELS[m]['dimensions']:<20}"
print(row)

# Top-3 Accuracy
row = f"{'Top-3 Accuracy':<25}"
for m in SELECTED_MODELS:
    row += f"{model_evals[m]['accuracy']:<20.0%}"
print(row)

# Avg Top Score
row = f"{'Avg Top Score':<25}"
for m in SELECTED_MODELS:
    row += f"{model_evals[m]['avg_top_score']:<20.4f}"
print(row)

# Embed Time
row = f"{'Embed Time (corpus)':<25}"
for m in SELECTED_MODELS:
    row += f"{model_times[m]:<20.2f}s"
print(row)

# Cost per 1k tokens
row = f"{'Cost per 1k tokens':<25}"
for m in SELECTED_MODELS:
    row += f"${MODELS[m]['cost_per_1k_tokens']:<19}"
print(row)

### Visual comparison

A grouped bar chart makes differences immediately obvious.

In [ ]:
# Build comparison chart -- 3 panels: Accuracy, Latency, Cost
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

model_labels = SELECTED_MODELS
bar_colors = ["#2196F3", "#4CAF50", "#FF9800"][:len(model_labels)]
x = range(len(model_labels))

# Panel 1: Top-3 Accuracy
accuracies = [model_evals[m]["accuracy"] * 100 for m in model_labels]
axes[0].bar(x, accuracies, color=bar_colors, edgecolor="white")
axes[0].set_title("Top-3 Accuracy (%)")
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_labels, rotation=15, fontsize=8)
axes[0].set_ylim(0, 110)
for i, v in enumerate(accuracies):
    axes[0].text(i, v + 2, f"{v:.0f}%", ha="center", fontsize=10, fontweight="bold")

# Panel 2: Embed Latency
latencies = [model_times[m] for m in model_labels]
axes[1].bar(x, latencies, color=bar_colors, edgecolor="white")
axes[1].set_title("Corpus Embed Time (seconds)")
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_labels, rotation=15, fontsize=8)
for i, v in enumerate(latencies):
    axes[1].text(i, v + 0.05, f"{v:.2f}s", ha="center", fontsize=10, fontweight="bold")

# Panel 3: Cost per 1k tokens
costs = [MODELS[m]["cost_per_1k_tokens"] for m in model_labels]
axes[2].bar(x, costs, color=bar_colors, edgecolor="white")
axes[2].set_title("Cost per 1k Tokens ($)")
axes[2].set_xticks(x)
axes[2].set_xticklabels(model_labels, rotation=15, fontsize=8)
for i, v in enumerate(costs):
    axes[2].text(i, v + 0.005, f"${v}", ha="center", fontsize=10, fontweight="bold")

fig.suptitle("Cross-Model Comparison: Accuracy vs Latency vs Cost", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### Per-query detail view

Where do models agree? Where do they disagree? This tells you more than the aggregate numbers.

In [ ]:
# Per-query hit/miss comparison
print(f"{'Query':<55}", end="")
for m in SELECTED_MODELS:
    print(f"{m:<16}", end="")
print()
print("-" * (55 + 16 * len(SELECTED_MODELS)))

for i, q in enumerate(test_queries):
    short_q = q["query"][:52] + "..." if len(q["query"]) > 52 else q["query"]
    print(f"{short_q:<55}", end="")
    for m in SELECTED_MODELS:
        hit = model_evals[m]["results"][i]["hit"]
        status = "HIT" if hit else "MISS"
        print(f"{status:<16}", end="")
    print()

---

## 7. Model Recommendation

Based on the data above, which model would you recommend for the Northbrook Partners pipeline?

Consider:
- **Accuracy**: Did the more expensive model actually retrieve better results for your queries?
- **Cost**: The Northbrook corpus is internal documents -- maybe a few hundred. How much does the cost difference matter at that scale?
- **Latency**: For an internal tool, is a small latency difference noticeable?
- **Future growth**: If the corpus grows to thousands of documents, does your choice change?

There is no single right answer. The skill is knowing how to **justify** the choice with data.

### Corpus cost estimate

If the Northbrook corpus has approximately 50 documents averaging 2,000 tokens each, what does it cost to embed the full corpus with each model?

In [ ]:
# Cost estimate for the full Northbrook corpus
NUM_DOCS = 50
AVG_TOKENS = 2000
TOTAL_TOKENS = NUM_DOCS * AVG_TOKENS

print(f"Estimated corpus: {NUM_DOCS} docs x {AVG_TOKENS} tokens = {TOTAL_TOKENS:,} tokens")
print()

for model_name in SELECTED_MODELS:
    cost_per_token = MODELS[model_name]["cost_per_1k_tokens"] / 1000
    total_cost = TOTAL_TOKENS * cost_per_token
    print(f"  {model_name:<20} ${total_cost:.4f} to embed full corpus")

print("\nThese are one-time costs (you embed the corpus once and store the vectors).")
print("Query costs are per-query and accumulate over time.")

### Your recommendation

Fill in the cell below with your model choice and justification. Use your evaluation data to support your decision.

## My Recommendation

**Model choice:** [Write your chosen model here]

**Justification:**

[Write 2-3 sentences explaining your choice based on the data above. Consider: accuracy difference, cost difference, latency difference, and the specific needs of the Northbrook pipeline. Reference specific numbers from your comparison table.]

---

## 8. Bridge to Ingestion

You can now turn text into vectors and measure which embedding model does the best job for your data. But there is a problem we have not solved yet.

Right now, we embedded the first 500 characters of each document. Real documents are pages long. You cannot embed a 20-page policy as a single vector -- too much meaning gets compressed into too few dimensions.

**The chunking problem:** How do you split a document into pieces without losing meaning? Where do you cut? What gets lost at the seams?

And once you have chunks, where do you store them so you can search thousands of them in milliseconds?

That is Session 2.2: **Chunking and Vector Ingestion with ChromaDB.**

### Questions to sit with:

> **On chunking:** "If you split a document into chunks, how do you decide where to cut? What gets lost at the seams?"

> **On metadata:** "When you retrieve a chunk, what else do you wish you knew about it?"

> **On scale:** "How would you organize 10,000 chunks so you can find the right 5?"

The embedding module you used today (`src/s2_embeddings/embed.py`) will be used directly in Session 2.2 when we chunk documents and ingest them into ChromaDB. The model you recommended today is the model you will use for your vector store.

---

## 9. Session Recap

### What you accomplished today:

- Generated your first embeddings and inspected the vectors
- Computed semantic similarity between sentences and saw that meaning, not keywords, drives the scores
- Visualized semantic clusters in a scatter plot and watched phrases group by topic
- Compared multiple Voyage AI models on a controlled retrieval task
- Built a comparison table and visual chart of accuracy, latency, and cost
- Made a data-driven model recommendation for the Northbrook pipeline

### Three takeaways:

1. **Embeddings turn meaning into math.** That is the foundation for everything we build next.
2. **There is no universally best model.** The right choice depends on your data, budget, and quality bar.
3. **The skill is knowing how to test.** You now have a repeatable method for evaluating any embedding model on any retrieval task.

### Homework (not graded, 30-45 min):

1. Add **5 more test queries** (including 2 that use different vocabulary than the source docs)
2. Re-run the comparison with the expanded query set
3. Write a **failure analysis** cell: which queries did both models get wrong? Why?
4. **Estimate corpus cost** for 50 documents averaging 2,000 tokens each (cell provided above)

This feeds directly into your Lab 1 work.

### Next session: Chunking and Vector Ingestion